## SETUP


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def load_data(path):
    return pd.read_csv(path)

def ber_pct(df):
    ok = (df['ber'] == '1.0').sum()
    total = (df['ber'] != '-').sum()
    return ok, total, ok / total * 100 if total else 0.0

def ber_uncond_pct(df):
    return (df['ber'] == '1.0').mean() * 100


### Load result to data frame


In [ ]:
single = load_data('single/result.csv')
multi = load_data('multi/result.csv')


# RAW BER — All Entries


#### Overall BER


In [ ]:
for label, df in [('Single', single), ('Multi', multi)]:
    ok, total, pct = ber_pct(df)
    uncond = ber_uncond_pct(df)
    print(f"{label}:  {ok}/{total} = {pct:.1f}% (conditional on CSR pass)")
    print(f"{label}:  {uncond:.1f}% (unconditional, '-' counted as fail)")
    print()


#### Per-Intent BER


In [ ]:
def intent_ber(df):
    ok = df[df['ber'] == '1.0'].groupby('intent').size()
    compiled = df[df['ber'] != '-'].groupby('intent').size()
    return (ok / compiled * 100).round(1).fillna(0)

ber_intent = pd.DataFrame({
    'Single': intent_ber(single),
    'Multi': intent_ber(multi),
})
print(ber_intent)

ax = ber_intent.plot.barh(figsize=(10, 6))
ax.set_title('Behavioral Equivalence Rate — Raw (Conditional on CSR Pass)')
ax.set_xlabel('BER (%)')
plt.tight_layout()
plt.show()


#### BER by Difficulty


In [ ]:
def diff_ber(df):
    ok = df[df['ber'] == '1.0'].groupby('difficulty').size()
    compiled = df[df['ber'] != '-'].groupby('difficulty').size()
    return (ok / compiled * 100).round(2).fillna(0)

ber_diff = pd.DataFrame({
    'Single': diff_ber(single),
    'Multi': diff_ber(multi),
})
print(ber_diff)


# BER — SUCCESS Exit Only


#### Overall BER (SUCCESS Only)


In [ ]:
s_valid = single[single['exit_status'] == 'SUCCESS']
m_valid = multi[multi['exit_status'] == 'SUCCESS']

for label, df in [('Single', s_valid), ('Multi', m_valid)]:
    ok, total, pct = ber_pct(df)
    print(f"{label}:  {ok}/{total} = {pct:.1f}%")


#### Per-Intent BER (SUCCESS Only)


In [ ]:
s_valid = single[single['exit_status'] == 'SUCCESS']
m_valid = multi[multi['exit_status'] == 'SUCCESS']

ber_succ = pd.DataFrame({
    'Single': intent_ber(s_valid),
    'Multi': intent_ber(m_valid),
})
print(ber_succ)

ax = ber_succ.plot.barh(figsize=(10, 6))
ax.set_title('Behavioral Equivalence Rate — SUCCESS Exit Only')
ax.set_xlabel('BER (%)')
plt.tight_layout()
plt.show()


#### BER by Difficulty (SUCCESS Only)


In [ ]:
s_valid = single[single['exit_status'] == 'SUCCESS']
m_valid = multi[multi['exit_status'] == 'SUCCESS']

ber_diff_succ = pd.DataFrame({
    'Single': diff_ber(s_valid),
    'Multi': diff_ber(m_valid),
})
print(ber_diff_succ)
